In [1]:
!pip install zarr -q --no-index --find-links=/kaggle/input/zarr-package/

In [18]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import polars as pl
import zarr
from zarr import ProcessSynchronizer
from zarr import DirectoryStore
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn import set_config
import torch
from tqdm import tqdm, trange
from numpy.typing import NDArray
from functools import lru_cache

set_config(transform_output = "default")

store = DirectoryStore('/kaggle/working/symbol_data.zarr')
synchronizer = ProcessSynchronizer('/kaggle/working/sync.lock')
root = zarr.group(store, overwrite=True, synchronizer=synchronizer)
symb_last_dates = {k:0 for k in range(39)}
latest_time_idx = 0 
categories = {"feature_09": [2, 4, 9, 11, 12, 14, 15, 25, 26, 30, 34, 42, 44, 46, 49, 50, 57, 64, 68, 70, 81, 82], "feature_10": [1, 2, 3, 4, 5, 6, 7, 10, 12], "feature_11": [9, 11, 13, 16, 24, 25, 34, 40, 48, 50, 59, 62, 63, 66, 76, 150, 158, 159, 171, 195, 214, 230, 261, 297, 336, 376, 388, 410, 522, 534, 539], "time_id": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 512, 513, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 524, 525, 526, 527, 528, 529, 530, 531, 532, 533, 534, 535, 536, 537, 538, 539, 540, 541, 542, 543, 544, 545, 546, 547, 548, 549, 550, 551, 552, 553, 554, 555, 556, 557, 558, 559, 560, 561, 562, 563, 564, 565, 566, 567, 568, 569, 570, 571, 572, 573, 574, 575, 576, 577, 578, 579, 580, 581, 582, 583, 584, 585, 586, 587, 588, 589, 590, 591, 592, 593, 594, 595, 596, 597, 598, 599, 600, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 612, 613, 614, 615, 616, 617, 618, 619, 620, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 631, 632, 633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 647, 648, 649, 650, 651, 652, 653, 654, 655, 656, 657, 658, 659, 660, 661, 662, 663, 664, 665, 666, 667, 668, 669, 670, 671, 672, 673, 674, 675, 676, 677, 678, 679, 680, 681, 682, 683, 684, 685, 686, 687, 688, 689, 690, 691, 692, 693, 694, 695, 696, 697, 698, 699, 700, 701, 702, 703, 704, 705, 706, 707, 708, 709, 710, 711, 712, 713, 714, 715, 716, 717, 718, 719, 720, 721, 722, 723, 724, 725, 726, 727, 728, 729, 730, 731, 732, 733, 734, 735, 736, 737, 738, 739, 740, 741, 742, 743, 744, 745, 746, 747, 748, 749, 750, 751, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766, 767, 768, 769, 770, 771, 772, 773, 774, 775, 776, 777, 778, 779, 780, 781, 782, 783, 784, 785, 786, 787, 788, 789, 790, 791, 792, 793, 794, 795, 796, 797, 798, 799, 800, 801, 802, 803, 804, 805, 806, 807, 808, 809, 810, 811, 812, 813, 814, 815, 816, 817, 818, 819, 820, 821, 822, 823, 824, 825, 826, 827, 828, 829, 830, 831, 832, 833, 834, 835, 836, 837, 838, 839, 840, 841, 842, 843, 844, 845, 846, 847, 848, 849, 850, 851, 852, 853, 854, 855, 856, 857, 858, 859, 860, 861, 862, 863, 864, 865, 866, 867, 868, 869, 870, 871, 872, 873, 874, 875, 876, 877, 878, 879, 880, 881, 882, 883, 884, 885, 886, 887, 888, 889, 890, 891, 892, 893, 894, 895, 896, 897, 898, 899, 900, 901, 902, 903, 904, 905, 906, 907, 908, 909, 910, 911, 912, 913, 914, 915, 916, 917, 918, 919, 920, 921, 922, 923, 924, 925, 926, 927, 928, 929, 930, 931, 932, 933, 934, 935, 936, 937, 938, 939, 940, 941, 942, 943, 944, 945, 946, 947, 948, 949, 950, 951, 952, 953, 954, 955, 956, 957, 958, 959, 960, 961, 962, 963, 964, 965, 966, 967], "symbol_id": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38]}

In [16]:
lags = pl.read_parquet('/kaggle/input/jane-street-real-time-market-data-forecasting/lags.parquet')
test = pl.read_parquet('/kaggle/input/jane-street-real-time-market-data-forecasting/test.parquet')

In [4]:
train_data = pl.scan_parquet('/kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet')

In [40]:
def format_dec_data(dec_tensor: list[torch.Tensor]):
    dec_tensor_ = torch.stack(dec_tensor)
    return {
        "decoder_length": torch.tensor(dec_tensor_.shape[0]).to(dtype=torch.int16),
        "decoder_categoricals": dec_tensor_[:, -5:].to(dtype=torch.int16).long(),
        "decoder_reals": dec_tensor_[:, :-5].to(dtype=torch.float16),
    }


def merge_current_data(
    curr_day_dec_store: dict[int, list[torch.Tensor]], timestep: torch.Tensor
):
    for data_idx in range(timestep.size(0)):
        symbol_id = timestep[data_idx, -1].item()
        symbol_data = timestep[data_idx, :]
        if symbol_id not in curr_day_dec_store:
            curr_day_dec_store[symbol_id] = [symbol_data]
        else:
            curr_day_dec_store[symbol_id].append(symbol_data)
    return curr_day_dec_store


def format_enc_data(data: torch.Tensor):
    device = data.device
    return {
        "encoder_length": torch.tensor(data.shape[0]).to(dtype=torch.int16, device=device),
        "encoder_categoricals": data[:, -5:].to(dtype=torch.int16, device=device).long(),
        "encoder_reals": data[:, :78].to(dtype=torch.float16, device=device),
        "encoder_targets": data[:, :-5][:, -9:].to(dtype=torch.float16, device=device),
    }


class DataStore:
    def __init__(self, *args, **kwargs):
        self.curr_day_dec_store: dict[str, list[torch.Tensor]] = {}
        self.store = DirectoryStore("/kaggle/working/symbol_data.zarr")
        self.synchronizer = ProcessSynchronizer("/kaggle/working/sync.lock")
        self.root = zarr.group(store, overwrite=True, synchronizer=synchronizer)
        self.symb_last_dates = {k: 0 for k in range(39)}
        self.latest_time_idx = 0
        self.categories = {
            "feature_09": [2, 4, 9, 11, 12, 14, 15, 25, 26, 30, 34, 42, 44, 46, 49, 50, 57, 64, 68, 70, 81, 82],
            "feature_10": [1, 2, 3, 4, 5, 6, 7, 10, 12],
            "feature_11": [9, 11, 13, 16, 24, 25, 34, 40, 48, 50, 59, 62, 63, 66, 76, 150, 158, 159, 171, 195, 214, 230, 261, 297, 336, 376, 388, 410, 522, 534, 539],
            "time_id": [i for i in range(968)],
            "symbol_id": [i for i in range(39)],
        }
        self.symbol_idx = 3
        self.time_id_idx = 2
        self.weight_idx = 4
        self.date_id_scaler = StandardScaler()
        self.date_id_scaler.fit(np.arange(0, 1900).reshape(-1, 1))
        self.time_idx_scaler = StandardScaler()
        self.time_idx_scaler.fit(np.arange(0, 1854468).reshape(-1, 1))
        self.real_scaler = StandardScaler()
        self.feat_idx = [ 4, 5, 6, 7, 8, 9, 10, 11, 12, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82]
        self.dec_ct = ColumnTransformer(
            transformers=[
                ("real_time_indices", "passthrough", [0, 1]),
                ("weight_idx", "passthrough", [self.weight_idx]),
                ("features", StandardScaler(), self.feat_idx),
                (
                    "cat_encoder",
                    OrdinalEncoder(
                        categories=[
                            categories["feature_09"],
                            categories["feature_10"],
                            categories["feature_11"],
                        ],
                        handle_unknown="use_encoded_value",
                        unknown_value=95,
                    ),
                    [14, 15, 16],
                ),
                ("static_real", "passthrough", [self.time_id_idx, self.symbol_idx]),
            ],
            remainder="drop",
        )
        self.tensor_device = "cpu"
        self.old_symb_order = []
        self.symb_order_changed = False
        self.curr_symb_order = []
        self.new_symb_order = []
        self.curr_day_enc_data = None
        self.pred_device = "cuda"

    def prep_dec_df(self, timestep: pl.DataFrame) -> torch.Tensor:
        symb_scaled = self.dec_ct.fit_transform(timestep.to_numpy())
        symb_scaled[:, 0] = self.time_idx_scaler.transform(
            symb_scaled[:, 0].reshape(-1, 1)
        ).ravel()
        symb_scaled[:, 1] = self.date_id_scaler.transform(
            symb_scaled[:, 1].reshape(-1, 1)
        ).ravel()
        return torch.tensor(symb_scaled, dtype=torch.float16, device=self.tensor_device)

    def add_time_idx(self, timestep: pl.DataFrame) -> pl.DataFrame:
        curr_date = timestep["date_id"].max()
        if curr_date < 677:
            timestep["date_id"] = timestep.with_columns(pl.col("date_id") + 1699)
        return (
            timestep.with_columns(
                (
                    (pl.col("date_id").cast(pl.Int32) - 677) * 968
                    + pl.col("time_id").cast(pl.Int32)
                    + (677 * 849)
                ).alias("time_idx")
            )
            .fill_null(0)
            .select(
                pl.col("time_idx"),
                pl.col(
                    [
                        x
                        for x in timestep.columns
                        if x not in ["time_idx", "partition_id"]
                    ]
                ),
            )
        )

    def flush_curr_day_data(self, lags: pl.DataFrame):
        if len(self.curr_day_dec_store) ==0:
            return False, "No Data to Flush"
        symb_lags = lags.sort("time_id").partition_by("symbol_id")
        curr_date = None
        for symb_lag in symb_lags:
            self.symb_last_dates[symb_lag["symbol_id"].min()] = symb_lag[
                "date_id"
            ].min()
            curr_symbol = symb_lag["symbol_id"].min()
            if curr_date is None:
                curr_date = symb_lag["date_id"].min()
            symb_lag = (
                symb_lag.select("^responder_.*$")
                .to_torch()
                .to(dtype=torch.float16, device=self.tensor_device)
            )
            combined_data = torch.cat(
                [torch.stack(self.curr_day_dec_store[curr_symbol],dim=0).cpu(), symb_lag], dim=-1
            )
            self.root[f"{curr_symbol}/{curr_date}"] = zarr.array(
                combined_data.cpu().numpy(), chunks=(968, 93), dtype=np.float16
            )
        self.curr_day_dec_store = {}
        self.curr_symb_order = []
        self.get_prev_data.cache_clear()
        self.curr_day_enc_data = None
        self.curr_batch_enc_data = None

    def update_curr_day_timestep(self, timestep: pl.DataFrame):
        # Since Everything is arriving in order no need for sorting
        timestep = self.add_time_idx(timestep)
        timestep: torch.Tensor = self.prep_dec_df(timestep)
        curr_symb_order = timestep[:, -4].numpy().tolist()
        if self.curr_symb_order != curr_symb_order:
            self.symb_order_changed = True
            self.new_symb_order = curr_symb_order
            self.old_symb_order = self.curr_symb_order
            self.curr_symb_order = curr_symb_order
        else:
            self.symb_order_changed = False
        self.curr_day_dec_store = merge_current_data(self.curr_day_dec_store, timestep)

    @lru_cache(maxsize=None)
    def get_prev_data(self, symbol_id: int, sampling=2):
        data = torch.tensor(
            self.root[f"{int(symbol_id)}/{self.symb_last_dates[symbol_id]}"].oindex[::sampling],
            dtype=torch.float16, device=self.pred_device
        )
        data_dict = format_enc_data(data)
        return data_dict
    
    def get_curr_batch(self):
        if self.curr_day_enc_data is None:
            self.curr_day_enc_data = {x: self.get_prev_data(x) for x in self.curr_symb_order}
            self.curr_batch_enc_data = {
                "encoder_reals": torch.stack([self.curr_day_enc_data[x]["encoder_reals"] for x in self.curr_symb_order]),
                "encoder_length": torch.stack([self.curr_day_enc_data[x]["encoder_length"] for x in self.curr_symb_order]),
                "encoder_targets": torch.stack([self.curr_day_enc_data[x]["encoder_targets"] for x in self.curr_symb_order]),
                "encoder_categoricals": torch.stack([self.curr_day_enc_data[x]["encoder_categoricals"] for x in self.curr_symb_order]),
            }
        if self.symb_order_changed:
            self.curr_batch_enc_data = {
            "encoder_reals": torch.stack([self.curr_day_enc_data[x]["encoder_reals"] for x in self.curr_symb_order]),
            "encoder_length": torch.stack([self.curr_day_enc_data[x]["encoder_length"] for x in self.curr_symb_order]),
            "encoder_targets": torch.stack([self.curr_day_enc_data[x]["encoder_targets"] for x in self.curr_symb_order]),
            "encoder_categoricals": torch.stack([self.curr_day_enc_data[x]["encoder_categoricals"] for x in self.curr_symb_order]),
            }
        dec_data = {x:format_dec_data(self.curr_day_dec_store[x]) for x in self.curr_symb_order}
        self.curr_batch_enc_data.update({
            "decoder_reals": torch.stack([dec_data[x]["decoder_reals"] for x in self.curr_symb_order]).to(self.pred_device),
            "decoder_length": torch.stack([dec_data[x]["decoder_length"] for x in self.curr_symb_order]).to(self.pred_device),
            "decoder_categoricals": torch.stack([dec_data[x]["decoder_categoricals"] for x in self.curr_symb_order]).to(self.pred_device),
        })
        return self.curr_batch_enc_data

In [41]:
# At every iter we are served with a timestep so
last_day = train_data.filter(pl.col("date_id") == 1695).collect()

In [42]:
lags = last_day.select(
    [
        "date_id",
        "time_id",
        "symbol_id",
        "^responder_.*$",
    ]
)
test_data = last_day.drop('^responder_.*$')
test_data_timesteps = test_data.partition_by("time_id")


In [43]:
DS = DataStore()

for ts in tqdm(test_data_timesteps):
    DS.update_curr_day_timestep(ts)
DS.flush_curr_day_data(lags)

100%|██████████| 968/968 [00:04<00:00, 204.86it/s]


In [45]:
last_day_data = train_data.filter(pl.col("date_id") == 1696).collect()
last_day_lags = last_day_data.select(
    [
        "date_id",
        "time_id",
        "symbol_id",
        "^responder_.*$",
    ]
)
test_data = last_day_data.drop('^responder_.*$')
test_data_timesteps = test_data.partition_by("time_id", maintain_order=True)

In [46]:
%%time
for ts in tqdm(test_data_timesteps):
    DS.update_curr_day_timestep(ts)
    p = DS.get_curr_batch()
DS.flush_curr_day_data(last_day_lags)

100%|██████████| 968/968 [00:16<00:00, 58.94it/s] 


CPU times: user 25.1 s, sys: 585 ms, total: 25.6 s
Wall time: 16.6 s
